# 07 — Final Comparison & Findings

Consolidates every model in this project under one evaluation protocol
(He et al. NCF: 1 held-out positive vs 99 sampled negatives; Recall@10 = Hit@10, plus NDCG@10).

| Notebook | Model | Data |
|---|---|---|
| 03 | Popularity baseline | full 25M |
| 05 | Surprise SVD (Funk-SVD, explicit) | 2M sample |
| 05 | **implicit ALS (WMF)** | full 25M |
| 04 | **WMF from scratch (HKV/ALS, NumPy)** | 15K-user subsample |
| 06 | Cold-start: TMDB content projection | TMDB-covered subset |

In [1]:
import os, sys, time
os.environ['OPENBLAS_NUM_THREADS'] = '1'
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, scipy.sparse as sp
import recsys_utils as ru
from implicit.cpu.als import AlternatingLeastSquares
ART = '../artifacts'
train_mat = sp.load_npz(f'{ART}/train_mat.npz')
c = np.load(f'{ART}/cands.npz'); users, cands = c['users'], c['cands']
print('full-data eval set:', train_mat.shape, '|', len(users), 'users')

/home/racloop/Documents/Personal/coursera/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


full-data eval set: (162541, 55413) | 162495 users


## Headline result on the full 25M dataset

In [2]:
item_pop = np.asarray(train_mat.getnnz(axis=0)).ravel().astype(np.float32)
r_pop, n_pop = ru.score_metrics(ru.score_popularity(item_pop, cands), k=10)
als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=ru.ALPHA,
                              iterations=15, random_state=42)
als.fit(train_mat, show_progress=False)
r_als, n_als = ru.score_metrics(ru.score_als(als.user_factors, als.item_factors, users, cands), k=10)
print(f'{"Model (full 25M)":<24}{"Recall@10":>12}{"NDCG@10":>12}')
print('-'*48)
print(f'{"Popularity baseline":<24}{r_pop:>12.4f}{n_pop:>12.4f}')
print(f'{"implicit ALS (WMF)":<24}{r_als:>12.4f}{n_als:>12.4f}')
print(f'\nWMF lifts NDCG@10 by {100*(n_als-n_pop)/n_pop:.1f}% over the popularity baseline.')

Model (full 25M)           Recall@10     NDCG@10
------------------------------------------------
Popularity baseline           0.9251      0.6606
implicit ALS (WMF)            0.9714      0.8097

WMF lifts NDCG@10 by 22.6% over the popularity baseline.


## Full results table (across notebooks)

| Model | Recall@10 | NDCG@10 | Notes |
|---|---|---|---|
| Popularity baseline (25M) | 0.925 | 0.661 | high — sampled-negative artifact (Rendle 2020) |
| Surprise SVD (2M) | 0.514 | 0.307 | optimizes RMSE, not ranking → wrong task framing |
| **implicit ALS / WMF (25M)** | **0.971** | **0.810** | best; the production-grade library |
| WMF from scratch (15K users) | 0.926 | 0.697 | matches library (0.928/0.696) on same data ✓ |
| Cold-start content projection | 0.385 | 0.201 | vs 0.10 random → content rescues new items |

### What I'd say about these numbers
1. **WMF clearly beats popularity** on the personalized metric (NDCG@10 0.81 vs 0.66).
2. **My from-scratch ALS matches the library** to within 0.002 — evidence the HKV math is right.
3. **Surprise underperforms popularity**: rating-prediction (RMSE) ≠ top-N ranking. The task
   framing matters more than the algorithm.
4. **Sampled-negative metrics flatter everything** (popularity hits 0.92); the honest signal is
   the *gap* between models, and a full unsampled ranking would separate them further.
5. **Cold-start**: a content→latent projection takes brand-new movies from ~0.10 (random) to
   0.385 Recall@10 — collaborative filtering alone can't touch zero-interaction items.

### Honest limitations
- Offline sampled metrics, not an online A/B test (the real production signal).
- From-scratch ALS validated on a 15K-user subsample (pure-NumPy per-user loop doesn't scale to 162K).
- Single train/test leave-one-out split; no hyperparameter sweep on factors/alpha/reg.

### Natural next step → reinforcement learning
This is a *static* model: train once, serve. The real product question is sequential —
*which* item to show *now* to maximise long-run engagement, learning online from feedback.
That reframes recommendation as a **contextual bandit** (context = user history, arm = item,
reward = engagement), balancing exploration of new items against exploitation of known-good ones.